In [1]:
import glob, zarr, os, napari
import dask.array as da
from tqdm.auto import tqdm
from natsort import natsorted
import napari
import pandas as pd
SCALE_TUPLE = (0.165, 0.165)
SPLIT_IMAGE_KWS =  ['top', 'bot', 'left', 'right']

# Generating full image FOVs

In [3]:
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*max_proj.zarr')
# Filter and sort
zarr_fns = natsorted([
                        fn for fn in zarr_fns
                        if not any(x in fn for x in SPLIT_IMAGE_KWS)])
zarr_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr/rep_1_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_2/zarr/rep_1_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_3/zarr/rep_1_mouse_3_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_4/zarr/rep_1_mouse_4_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_5/zarr/rep_1_mouse_5_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/rep_2_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_7/zarr/rep_2_mouse_7_max_proj.zarr',
 '/mnt/O

In [2]:
df_metadata = pd.read_parquet('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/condition_metadata.parquet')

In [3]:
df_metadata

,mouse_N,rep_N,condition
0,1,1,H2O
1,2,1,DMSO
2,3,1,RIF
3,4,1,RIF
4,5,1,RIF
5,6,1,RIF
6,7,1,PZA
7,8,1,PZA
8,9,1,PZA
9,10,1,PZA


In [6]:
zarr_fn = zarr_fns[5]
print(zarr_fn)

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr


In [9]:
# Filter for PZA and RIF conditions
target_conditions = ['PZA', 'RIF']
target_conditions = ['RIF']
df_filtered = df_metadata[df_metadata['condition'].isin(target_conditions)]

# Generate file paths using f-strings
base_path = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice"
zarr_files = [
    f"{base_path}/rep{row.rep_N}/mouse_{row.mouse_N}/zarr/rep_{row.rep_N}_mouse_{row.mouse_N}_max_proj.zarr"
    for _, row in df_filtered.iterrows()
]

# Quick check
print(f"Found {len(zarr_files)} files.")
print(zarr_files[:2])

Found 9 files.
['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_3/zarr/rep_1_mouse_3_max_proj.zarr', '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_4/zarr/rep_1_mouse_4_max_proj.zarr']


In [13]:
import napari
import glob
import os
import json
import re
import dask.array as da
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

# Setup QC directory
qc_dir = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/tissue_figures/screenshots_v3")
qc_dir.mkdir(exist_ok=True)

mtb_colormap = {1: '#FFFF00'} # yellow: #FFFF00

# Regex to parse info from filename
rep_pattern = re.compile(r"rep_(\d+)")
mouse_pattern = re.compile(r"mouse_(\d+)")
    
for zarr_fn in zarr_files:

    # --- Parse Identity ---
    basename = os.path.basename(os.path.normpath(zarr_fn))
    
    rep_match = rep_pattern.search(basename)
    rep_id = int(rep_match.group(1)) if rep_match else None
    
    mouse_match = mouse_pattern.search(basename)
    mouse_id = int(mouse_match.group(1)) if mouse_match else None
    # --- Lookup Condition ---
    condition = "Unknown"
    if rep_id is not None and mouse_id is not None:
        # Query the metadata df for the matching row
        # keys: mouse_num, replicate, condition
        match = df_metadata[
            (df_metadata['mouse_N'] == mouse_id) & 
            (df_metadata['rep_N'] == rep_id)
        ]
        if not match.empty:
            condition = match.iloc[0]['condition']
    
    # --- Load Data ---
    pyramid_image_stack = [da.from_zarr(fn) for fn in glob.glob(f"{zarr_fn}/s*")]
    cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
    mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_mtb/0')
    
    # --- Viewer Setup ---
    viewer = napari.Viewer(title=basename)
    
    viewer.add_image(pyramid_image_stack, channel_axis=0, 
                     colormap=['blue', 'green', 'magenta'], scale=SCALE_TUPLE)
    viewer.add_labels(cyto_masks, name='cytoplasm', scale=SCALE_TUPLE,)# contour=4)
    viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
                      colormap=mtb_colormap,) #contour=4)
    
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "um"
    
    # --- Screenshot Keybinding ---
    @viewer.bind_key('s')
    def save_qc_screenshot(viewer):
        # Define base pattern
        base_name_str = f"{basename}_{condition}_QC"
        
        # Check for existence and increment counter if needed
        counter = 0
        candidate_name = base_name_str
        while (qc_dir / f"{candidate_name}.png").exists():
            counter += 1
            candidate_name = f"{base_name_str}_{counter}"
        
        # 1. Save Image
        image_path = qc_dir / f"{candidate_name}.png"
        viewer.screenshot(path=str(image_path))
        
        # 2. Save Metadata
        meta = {
            "file": basename,
            "condition": condition,
            "mouse": mouse_id,
            "rep": rep_id,
            "zoom": viewer.camera.zoom,
            "center": list(viewer.camera.center),
            "layers": {l.name: l.visible for l in viewer.layers},
            "save_iteration": counter
        }
        
        json_path = qc_dir / f"{candidate_name}.json"
        with open(json_path, 'w') as f:
            json.dump(meta, f, indent=4)
            
        viewer.status = f"Saved: {candidate_name}"
        print(f"Captured: {candidate_name}")
    viewer.show(block=True)

ArrayNotFoundError: No array found in store file:////snap at path 

# V3 of tissue slices

In [5]:
import re
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr')

# Filter for: (rep1 AND mouse_5) OR (rep1 AND mouse_10) OR (rep2 AND mouse_1)
targets = [
    ('/rep1/', '/mouse_5/'),
    ('/rep1/', '/mouse_10/'),
    ('/rep2/', '/mouse_1/')
]

zarr_fns = [
    f for f in zarr_fns 
    if any(rep in f and mouse in f for rep, mouse in targets)
]

In [6]:
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr')

for i, fn in enumerate(zarr_fns):
    print(i, os.path.basename(fn))

0 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr
1 rep_2_mouse_1_max_proj.zarr
2 20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555.zarr
3 rep_1_mouse_5_max_proj.zarr
4 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
5 rep_1_mouse_10_left_max_proj.zarr
6 rep_1_mouse_10_right_max_proj.zarr


In [7]:
zarr_fns = [zarr_fns[i] for i in [1, 3, 4]]

In [8]:
zarr_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_5/zarr/rep_1_mouse_5_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_10/zarr/20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr']

In [9]:
import napari
import glob
import os
import json
import re
import dask.array as da
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

# Setup QC directory
qc_dir = Path("/mnt/OPERA3/Nathan/illustration/dt_manuscript_figures_2026/tissue_figures/screenshots_v3")
qc_dir.mkdir(exist_ok=True)

mtb_colormap = {1: '#FFFF00'} # yellow: #FFFF00

# Regex to parse info from filename
rep_pattern = re.compile(r"rep_(\d+)")
mouse_pattern = re.compile(r"mouse_(\d+)")
    
for zarr_fn in zarr_fns-:
    # --- Load Data ---
    # if not os.path.exists(zarr_fn):
    #     # Fallback: look in the 'zarr' parent directory for any .zarr not containing 'rep'
    #     parent_dir = os.path.dirname(zarr_fn)
    #     fallback_options = [
    #         f for f in glob.glob(f"{parent_dir}/*.zarr") 
    #         if "rep" not in os.path.basename(f)
    #     ]
        
    #     if fallback_options:
    #         zarr_fn = fallback_options[0]
    #         pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]
    #         ch_ax=1
    #         print(f"Original path not found. Falling back to: {zarr_fn}")
    #     else:
    #         print(f"Warning: Could not find original or fallback Zarr for {zarr_fn}")
    #         continue
    # try:
    #     pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/s*"))]
    #     ch_ax=0
    # except:
    pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]
    ch_ax=1
        


    # cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
    # mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_mtb/0')
    # --- Parse Identity ---
    basename = os.path.basename(os.path.normpath(zarr_fn))
    
    rep_match = rep_pattern.search(basename)
    rep_id = int(rep_match.group(1)) if rep_match else None
    
    mouse_match = mouse_pattern.search(basename)
    mouse_id = int(mouse_match.group(1)) if mouse_match else None
    # --- Lookup Condition ---
    condition = "Unknown"
    if rep_id is not None and mouse_id is not None:
        # Query the metadata df for the matching row
        # keys: mouse_num, replicate, condition
        match = df_metadata[
            (df_metadata['mouse_N'] == mouse_id) & 
            (df_metadata['rep_N'] == rep_id)
        ]
        if not match.empty:
            condition = match.iloc[0]['condition']
    
    # --- Load Data ---
    # pyramid_image_stack = [da.from_zarr(fn) for fn in glob.glob(f"{zarr_fn}/s*")]
    # cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
    # mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_mtb/0')
    
    # --- Viewer Setup ---
    viewer = napari.Viewer(title=basename)
    
    viewer.add_image(pyramid_image_stack, channel_axis=ch_ax, 
                     colormap=['blue', 'green', 'magenta'], scale=SCALE_TUPLE)
    # viewer.add_labels(cyto_masks, name='cytoplasm', scale=SCALE_TUPLE,)# contour=4)
    # viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
    #                   colormap=mtb_colormap,) #contour=4)
    
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "um"
    
    @viewer.bind_key('s')
    def save_qc_screenshot(viewer):
        # Define base pattern and handle incrementing
        base_name_str = f"{basename}_{condition}_QC"
        counter = 0
        while any(qc_dir.glob(f"{base_name_str}_{counter}*")):
            counter += 1
        
        # Prefix for this batch of screenshots
        batch_prefix = f"{base_name_str}_{counter}"
        
        # Store original visibility states to restore later
        original_visibility = {l.name: l.visible for l in viewer.layers}
    
        # Helper to save both PNG and JSON
        def capture(name_suffix):
            candidate_name = f"{batch_prefix}_{name_suffix}"
            image_path = qc_dir / f"{candidate_name}.png"
            viewer.screenshot(path=str(image_path))
            
            meta = {
                "file": basename,
                "condition": condition,
                "mouse": mouse_id,
                "rep": rep_id,
                "zoom": viewer.camera.zoom,
                "center": list(viewer.camera.center),
                "view_type": name_suffix,
                "save_iteration": counter
            }
            with open(qc_dir / f"{candidate_name}.json", 'w') as f:
                json.dump(meta, f, indent=4)
    
        # 1. Capture All Channels (as currently visible)
        capture("all_channels")
    
        # 2. Capture Individual Channels
        # Turn everything off first
        for l in viewer.layers:
            l.visible = False
    
        # Toggle each one on, capture, then off again
        for l in viewer.layers:
            l.visible = True
            # Clean name for filename (remove spaces/slashes)
            clean_layer_name = l.name.replace(" ", "_").replace("/", "-")
            capture(f"channel_{clean_layer_name}")
            l.visible = False
    
        # Restore original visibility
        for l in viewer.layers:
            l.visible = original_visibility[l.name]
                
        viewer.status = f"Saved batch: {batch_prefix}"
        print(f"Captured batch: {batch_prefix}")
    viewer.show(block=True)

Captured batch: rep_2_mouse_1_max_proj.zarr_H2O_QC_0
Captured batch: rep_2_mouse_1_max_proj.zarr_H2O_QC_1
Captured batch: rep_2_mouse_1_max_proj.zarr_H2O_QC_2
Captured batch: rep_2_mouse_1_max_proj.zarr_H2O_QC_4


AttributeError: 'list' object has no attribute 'shape'

In [11]:
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr')

for i, fn in enumerate(zarr_fns):
    print(i, os.path.basename(fn))

0 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr
1 rep_2_mouse_1_max_proj.zarr
2 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5573.zarr
3 rep_2_mouse_2_max_proj.zarr
4 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5572.zarr
5 rep_2_mouse_3_bot_max_proj.zarr
6 rep_2_mouse_3_top_max_proj.zarr
7 20251014_40X_TimerMtb_BP_rep2_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5587.zarr
8 rep_2_mouse_4_bot_max_proj.zarr
9 rep_2_mouse_4_top_max_proj.zarr
10 20251014_40X_TimerMtb_BP_rep2_mice5_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5589.zarr
11 rep_2_mouse_5_bot_max_proj.zarr
12 rep_2_mouse_5_top_max_proj.zarr
13 20251030_40X_TimerMtb_BP_rep2_mice11_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251106_5690.zarr
14 rep_2_mouse_6_bot_max_proj.zarr
15 rep_2_mouse_6_top_max_proj.zarr
16 20251007_40X_TimerMtb_BP_rep2_mic

In [19]:
import napari
import glob
import os
import json
import re
import dask.array as da
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

# Setup QC directory
qc_dir = Path("/mnt/OPERA3/Nathan/illustration/dt_manuscript_figures_2026/tissue_figures/screenshots_v3")
qc_dir.mkdir(exist_ok=True)

mtb_colormap = {1: '#FFFF00'} # yellow: #FFFF00

# Regex to parse info from filename
rep_pattern = re.compile(r"rep_(\d+)")
mouse_pattern = re.compile(r"mouse_(\d+)")


zarr_fn = zarr_fns[39]
# zarr_fn = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_10/zarr/20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr'
# for zarr_fn in zarr_fns-:
    # --- Load Data ---
    # if not os.path.exists(zarr_fn):
    #     # Fallback: look in the 'zarr' parent directory for any .zarr not containing 'rep'
    #     parent_dir = os.path.dirname(zarr_fn)
    #     fallback_options = [
    #         f for f in glob.glob(f"{parent_dir}/*.zarr") 
    #         if "rep" not in os.path.basename(f)
    #     ]
        
    #     if fallback_options:
    #         zarr_fn = fallback_options[0]
    #         pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]
    #         ch_ax=1
    #         print(f"Original path not found. Falling back to: {zarr_fn}")
    #     else:
    #         print(f"Warning: Could not find original or fallback Zarr for {zarr_fn}")
    #         continue
    # try:
    #     pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/s*"))]
    #     ch_ax=0
    # except:
# pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]
# ch_ax=1
    
pyramid_image_stack = [da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/s*"))]
ch_ax=0

# cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_bot/0')
# --- Parse Identity ---
basename = os.path.basename(os.path.normpath(zarr_fn))

rep_match = rep_pattern.search(basename)
rep_id = int(rep_match.group(1)) if rep_match else None

mouse_match = mouse_pattern.search(basename)
mouse_id = int(mouse_match.group(1)) if mouse_match else None
# --- Lookup Condition ---
condition = "Unknown"
if rep_id is not None and mouse_id is not None:
    # Query the metadata df for the matching row
    # keys: mouse_num, replicate, condition
    match = df_metadata[
        (df_metadata['mouse_N'] == mouse_id) & 
        (df_metadata['rep_N'] == rep_id)
    ]
    if not match.empty:
        condition = match.iloc[0]['condition']

# --- Viewer Setup ---
viewer = napari.Viewer(title=basename)

viewer.add_image(pyramid_image_stack, channel_axis=ch_ax, 
                 colormap=['blue', 'green', 'magenta'], scale=SCALE_TUPLE)
# viewer.add_labels(cyto_masks, name='cytoplasm', scale=SCALE_TUPLE,)# contour=4)
viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
                  colormap=mtb_colormap,) #contour=4)

viewer.scale_bar.visible = True
viewer.scale_bar.unit = "um"

@viewer.bind_key('s')
def save_qc_screenshot(viewer):
    # Define base pattern and handle incrementing
    base_name_str = f"{basename}_{condition}_QC"
    counter = 0
    while any(qc_dir.glob(f"{base_name_str}_{counter}*")):
        counter += 1
    
    # Prefix for this batch of screenshots
    batch_prefix = f"{base_name_str}_{counter}"
    
    # Store original visibility states to restore later
    original_visibility = {l.name: l.visible for l in viewer.layers}

    # Helper to save both PNG and JSON
    def capture(name_suffix):
        candidate_name = f"{batch_prefix}_{name_suffix}"
        image_path = qc_dir / f"{candidate_name}.png"
        viewer.screenshot(path=str(image_path))
        
        meta = {
            "file": basename,
            "condition": condition,
            "mouse": mouse_id,
            "rep": rep_id,
            "zoom": viewer.camera.zoom,
            "center": list(viewer.camera.center),
            "view_type": name_suffix,
            "save_iteration": counter
        }
        with open(qc_dir / f"{candidate_name}.json", 'w') as f:
            json.dump(meta, f, indent=4)

    # 1. Capture All Channels (as currently visible)
    capture("all_channels")

    # 2. Capture Individual Channels
    # Turn everything off first
    for l in viewer.layers:
        l.visible = False

    # Toggle each one on, capture, then off again
    for l in viewer.layers:
        l.visible = True
        # Clean name for filename (remove spaces/slashes)
        clean_layer_name = l.name.replace(" ", "_").replace("/", "-")
        capture(f"channel_{clean_layer_name}")
        l.visible = False

    # Restore original visibility
    for l in viewer.layers:
        l.visible = original_visibility[l.name]
            
    viewer.status = f"Saved batch: {batch_prefix}"
    print(f"Captured batch: {batch_prefix}")
    # viewer.show(block=True)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/utils/colormaps/colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (52070, 45850) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_0
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_1
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_2
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_3
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_4
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_5
Captured batch: rep_2_mouse_15_bot_max_proj.zarr_PZA_QC_6


In [15]:
viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
                  colormap=mtb_colormap,) #contour=4)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/utils/colormaps/colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (54144, 45850) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Labels layer 'mtb' at 0x7567ecdc82d0>

Captured batch: rep_1_mouse_6_bot_max_proj.zarr_RIF_QC_0
Captured batch: rep_1_mouse_6_bot_max_proj.zarr_RIF_QC_1
Captured batch: rep_1_mouse_6_bot_max_proj.zarr_RIF_QC_2
Captured batch: rep_1_mouse_6_bot_max_proj.zarr_RIF_QC_3


In [11]:
viewer = napari.Viewer()

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_qt/qt_viewer.py", line 1106, in _qt_open
    self.viewer.open(
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1405, in open
    layers = self._open_or_raise_error(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1526, in _open_or_raise_error
    raise MultipleReaderError(
napari.errors.reader_errors.MultipleReaderError: Multiple plugins found capable of reading /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_9/zarr/20251007_40X_TimerMtb_BP_rep2_mice7_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251008_5586.zarr. Select plugin from {'napari-ome-zarr': 'napari-ome-zarr', 'napari': 'napari builtins'} and pass to reading function e.g. `viewer.open(..., plugin=...)

In [1]:
[da.from_zarr(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]

NameError: name 'glob' is not defined

In [26]:
[print(fn) for fn in sorted(glob.glob(f"{zarr_fn}/0/*"))]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/0/0
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/0/1
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/0/2
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/0/3
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr/0/4
/mnt/OPERA

[None, None, None, None, None, None, None, None, None]

In [7]:
# --- Lookup Condition ---
condition = "Unknown"
if rep_id is not None and mouse_id is not None:
    # Query the metadata df for the matching row
    # keys: mouse_num, replicate, condition
    match = df_metadata[
        (df_metadata['mouse_num'] == mouse_id) & 
        (df_metadata['replicate'] == rep_id)
    ]
    if not match.empty:
        condition = match.iloc[0]['condition']


In [ ]:
viewer.scale_bar.

In [17]:
viewer.scale_bar.font_size = 0  # default is 10


Captured: rep_2_mouse_1_max_proj.zarr_H2O_QC_20


In [ ]:
 "zoom": 6.360214222434655,
    "center": [
        0.0,
        3080.904339020486,
        2191.582809159516
    ],

In [18]:
viewer.camera.center = (0, 3080.904339020486,
        2191.582809159516)

In [19]:
viewer.camera.zoom = 6.360214222434655

Captured: rep_2_mouse_1_max_proj.zarr_H2O_QC_21


### started at 1705

In [19]:
from datetime import datetime

# Get and print current time
print(datetime.now())

# Optional: Print just the time (HH:MM:SS)
print(datetime.now().strftime("%H:%M:%S"))

2025-12-22 11:22:02.779232
11:22:02
